In [249]:
print(f"ES API with Search_Engine Class")

ES API with Search_Engine Class


### Test all ES api with Search_Engine.py calss
* Use ES Api via "ES_client"

In [250]:
from elasticsearch import Elasticsearch
import os
import json
import pandas as pd
from dotenv import load_dotenv
from Search_Engine import Search
import warnings
warnings.filterwarnings("ignore")

In [251]:
''' pip install python-dotenv'''
# load_dotenv() # will search for .env file in local folder and load variables
# Reload the variables from your .env file, overriding existing ones
load_dotenv("../../.env", override=True)

False

In [252]:
def get_headers():
    ''' Elasticsearch Header '''
    return {
            'Content-type': 'application/json', 
            'Authorization' : '{}'.format(os.getenv('BASIC_AUTH')),
            # 'Connection': 'close'
    }

In [253]:
def get_es_instance(host):
    es_client = Elasticsearch(hosts="{}".format(host), headers=get_headers(), timeout=5,  verify_certs=False, ssl_show_warn=False)
    return es_client

In [254]:
''' Source cluster '''
# es_obj_s_client = get_es_instance("http://localhost:9201")
es_client = Search(["http://localhost:9201"])

In [255]:
# Get info of ES cluster
print(json.dumps(es_client.get_es_info(), indent=2))

2026-05-29 09:01:58,957 : INFO : GET http://localhost:9201/ [status:200 request:0.009s]


{
  "name": "fn-dm-bees-omni-data-01",
  "cluster_name": "docker-cluster",
  "cluster_uuid": "9dhpDIHTR7CobqEnNoqabA",
  "version": {
    "number": "5.6.4",
    "build_hash": "8bbedf5",
    "build_date": "2017-10-31T18:55:38.105Z",
    "build_snapshot": false,
    "lucene_version": "6.6.1"
  },
  "tagline": "You Know, for Search"
}


In [256]:
# Get version of ES cluster
print(es_client.get_es_info()['version']['number'])

2026-05-29 09:01:58,975 : INFO : GET http://localhost:9201/ [status:200 request:0.004s]


5.6.4


In [257]:
# GET health of ES cluster
# resp = es_obj_s_client.cluster.health()
# print(json.dumps(resp, indent=2))

print(json.dumps(es_client.get_es_client_health(), indent=2))

2026-05-29 09:01:58,998 : INFO : GET http://localhost:9201/_cat/health?format=json [status:200 request:0.005s]


[
  {
    "epoch": "1780063318",
    "timestamp": "14:01:58",
    "cluster": "docker-cluster",
    "status": "yellow",
    "node.total": "1",
    "node.data": "1",
    "shards": "202",
    "pri": "202",
    "relo": "0",
    "init": "0",
    "unassign": "201",
    "pending_tasks": "0",
    "max_task_wait_time": "-",
    "active_shards_percent": "50.1%"
  }
]


In [258]:
# /_cat/nodes?format=json&h=name,ip,h,diskTotal,diskUsed,diskAvail,diskUsedPercent
# Returns a list of dictionaries with specific node metrics
nodes_info = es_client.get_cat_nodes_stat()

for node in nodes_info:
    print(f"Node {node['name']} at {node['ip']} has role {node['node.role']}")

2026-05-29 09:01:59,035 : INFO : GET http://localhost:9201/_cat/nodes?format=json&h=ip%2Cname%2Cnode.role%2Cheap.percent [status:200 request:0.024s]


Node fn-dm-bees-omni-data-01 at 172.23.0.4 has role mdi


In [259]:
# /_nodes/stats
# Returns a list of dictionaries with specific node metrics
nodes_stats_info = es_client.get_nodes_stats()

print(f"nodes_stats : {nodes_stats_info}")
print(f"nodes total : {nodes_stats_info.get('_nodes').get('total')}")

for each_node_key, each_node_value in nodes_stats_info.get("nodes").items():
    print("*"*10)
    print(f"each_node_key : {each_node_key}")
    print(each_node_value.get("name"))
    print(each_node_value.get("roles"))
    print(f"CPU used percent : {each_node_value.get('process').get('cpu').get('percent')}%")
    print(f"JVM heap used percent : {each_node_value.get('jvm').get('mem').get('heap_used_percent')}%")
    print(f"File Disk : {each_node_value.get('fs')}")
    print("*"*10)


2026-05-29 09:01:59,071 : INFO : GET http://localhost:9201/_nodes/stats [status:200 request:0.019s]


nodes_stats : {'_nodes': {'total': 1, 'successful': 1, 'failed': 0}, 'cluster_name': 'docker-cluster', 'nodes': {'1WhkOBUMTFGpr6qxqiHiMQ': {'timestamp': 1780063319055, 'name': 'fn-dm-bees-omni-data-01', 'transport_address': '172.23.0.4:9300', 'host': '172.23.0.4', 'ip': '172.23.0.4:9300', 'roles': ['master', 'data', 'ingest'], 'attributes': {'ml.max_open_jobs': '10', 'ml.enabled': 'true'}, 'indices': {'docs': {'count': 276118, 'deleted': 597}, 'store': {'size_in_bytes': 111461350, 'throttle_time_in_millis': 0}, 'indexing': {'index_total': 350, 'index_time_in_millis': 620, 'index_current': 0, 'index_failed': 0, 'delete_total': 8, 'delete_time_in_millis': 24, 'delete_current': 0, 'noop_update_total': 0, 'is_throttled': False, 'throttle_time_in_millis': 0}, 'get': {'total': 1380, 'time_in_millis': 242, 'exists_total': 1380, 'exists_time_in_millis': 242, 'missing_total': 0, 'missing_time_in_millis': 0, 'current': 0}, 'search': {'open_contexts': 0, 'query_total': 1383, 'query_time_in_millis

In [260]:
''' sample '''
''' https://docs.kanaries.net/ko/topics/Pandas/pandas-add-column '''
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Age': [25, 30, 35, 40]
}
 
df = pd.DataFrame(data)
df.head(10)

,Name,Age
0,Alice,25
1,Bob,30
2,Charlie,35
3,David,40
